# F5-probability — Session 04: Conditional Probability and Bayes

**Session length:** about 85 minutes • **Concepts:** conditional-probability, bayes-rule.

We will restrict probability models to observed evidence, reverse a conditional with Bayes' rule, and keep every denominator honest. Checkpoint answers are collected at the end.


In [ ]:
import numpy as np


## 1. Joint tables: intersections before conditionals

Two events $A$ and $B$ split outcomes into four disjoint cells.

| | $B$ | $B^c$ | total |
| --- | ---: | ---: | ---: |
| $A$ | 18 | 12 | 30 |
| $A^c$ | 22 | 48 | 70 |
| total | 40 | 60 | 100 |

The $A\cap B$ cell is 18. Marginal totals live on the edges. Also,
$P(A\cup B)=P(A)+P(B)-P(A\cap B)$ because the intersection was counted twice.

### Checkpoint 1

1. Find $P(A)$, $P(B)$, $P(A\cap B)$, and $P(A\cup B)$.
2. Verify the union by adding its three joint cells.


## 2. Conditioning changes the denominator

For $P(B)>0$,

$$P(A\mid B)=\frac{P(A\cap B)}{P(B)}.$$

Read this as “among outcomes where $B$ happened.” The numerator is the intersection; the denominator is the total for the event after the bar. Thus $P(A\mid B)=18/40$, while $P(B\mid A)=18/30$. The numerator happens to match, but the directions and denominators do not.

The condition $P(B)>0$ is part of the definition. If $P(B)=0$, ordinary conditional probability is **undefined**. An empirical function must raise an error rather than return 0.

### Checkpoint 2

1. Reduce both conditionals above and explain why they differ.
2. What should code do if its conditioning mask contains no `True` entries?


## 3. Multiplication and chain rules

Rearranging the definition gives

$$P(A\cap B)=P(A\mid B)P(B)=P(B\mid A)P(A).$$

For stages,

$$P(A\cap B\cap C)=P(A)P(B\mid A)P(C\mid A\cap B),$$

provided every displayed conditioning event has positive probability.

### Checkpoint 3

A bag is red with probability $2/5$; conditional on red, a token is starred with probability $3/7$. Find the probability of “red and starred,” and state the validity condition for adding a third conditional stage.


## 4. Total probability counts every route

If $A_1,\ldots,A_k$ form a disjoint partition, then the events $B\cap A_i$ partition $B$:

$$P(B)=\sum_i P(B\mid A_i)P(A_i).$$

For the two-case split $A,A^c$,

$$P(B)=P(B\mid A)P(A)+P(B\mid A^c)P(A^c).$$

A detector alarms on 90% of faulty parts and 5% of sound parts; 2% of parts are faulty. Both the faulty-and-alarm route and the sound-and-alarm route belong in $P(\text{alarm})$.

### Checkpoint 4

Write the full total-probability expression for $P(\text{alarm})$ and label its two routes.


## 5. Bayes reverses the direction

Combine the multiplication rule with total probability:

$$P(A\mid B)=\frac{P(B\mid A)P(A)}
{P(B\mid A)P(A)+P(B\mid A^c)P(A^c)}.$$

$P(A)$ is the prior, $P(B\mid A)$ is a likelihood, and $P(A\mid B)$ is the posterior. The numerator is one route to evidence $B$; the denominator contains **all** routes to $B$. Bayes never licenses $P(A\mid B)=P(B\mid A)$.

### Checkpoint 5

For the detector above, write $P(\text{faulty}\mid\text{alarm})$ as one fraction. Before calculating, predict whether it is closer to 2%, 50%, or 90%.


In [ ]:
p_fault = 0.02
p_alarm_given_fault = 0.90
p_alarm_given_sound = 0.05
p_alarm = p_alarm_given_fault * p_fault + p_alarm_given_sound * (1 - p_fault)
p_fault_given_alarm = p_alarm_given_fault * p_fault / p_alarm
print("P(alarm):", p_alarm)
print("P(fault | alarm):", p_fault_given_alarm)


## 6. Empirical conditionals with Boolean masks

For paired observations, `event & given` is the intersection and `given.sum()` is the new denominator. Shape checks prevent accidental broadcasting.


In [ ]:
def conditional_rate(event, given):
    event = np.asarray(event, dtype=bool)
    given = np.asarray(given, dtype=bool)
    if event.ndim != 1 or event.shape != given.shape:
        raise ValueError("event and given must be same-length 1-D masks")
    denominator = int(given.sum())
    if denominator == 0:
        raise ValueError("conditioning event has zero observations")
    return int((event & given).sum()) / denominator

rain = np.array([1, 0, 1, 0, 0, 1, 0, 1], dtype=bool)
alert = np.array([1, 1, 1, 0, 0, 0, 0, 1], dtype=bool)
print("P(rain | alert):", conditional_rate(rain, alert))
print("P(alert | rain):", conditional_rate(alert, rain))


### Checkpoint 6

1. Explain why `(rain & alert).mean() / alert.mean()` equals the count ratio when the denominator is positive.
2. Predict the exception from conditioning on `np.zeros_like(rain)`. Why would returning 0 be false?


## 7. The base-rate trap and common pitfalls

A highly sensitive test can have a modest posterior when the target is rare. In 10,000 people at a 1% base rate, there are about 100 target cases but 9,900 non-target cases. Even a small false-positive rate acts on the much larger group.

- **Broken:** swap $P(A\mid B)$ and $P(B\mid A)$. **Fix:** name the denominator event.
- **Broken:** omit the $A^c$ route from Bayes' denominator. **Fix:** use total probability.
- **Broken:** condition on an impossible event. **Fix:** state positivity or raise an error.
- **Broken:** say “95% accurate” without separating sensitivity, false-positive rate, and prior. **Fix:** build the joint table.

### Checkpoint 7

A student says, “99% sensitivity means a positive gives 99% disease probability.” Identify the reversed conditional and name the two other rates required.


## 8. Worked exam-style example

A filter sees 4% malicious messages. It flags 92% of malicious messages and 3% of benign messages.

In a base of 10,000, there are 400 malicious and 9,600 benign messages. The two flagged cells are $0.92(400)=368$ and $0.03(9600)=288$, so

$$P(\text{malicious}\mid\text{flag})=\frac{368}{368+288}=\frac{23}{41}.$$

The posterior is not 92% because 92% is the reverse conditional and ignores benign false positives.

### Checkpoint 8

Change only the malicious base rate to 0.4%. Predict the posterior's direction of change and write its new total-probability denominator.


## 9. Exam connections and going deeper

Round 1 questions hide conditioning in words such as “given,” “among,” and “after observing.” Draw a two-by-two table, label the requested direction, and check that the denominator matches the event after the bar.

**Going deeper:** Session 05 quantifies reliability of empirical averages. Later model-evaluation units reuse these directions for precision and recall; this session supplies their probability foundation.


## Checkpoint Answers

<details><summary><b>Checkpoint 1</b></summary>
$P(A)=0.30$, $P(B)=0.40$, $P(A\cap B)=0.18$, and $P(A\cup B)=0.52=(18+12+22)/100$.
</details>

<details><summary><b>Checkpoint 2</b></summary>
$P(A\mid B)=9/20$ and $P(B\mid A)=3/5$. A zero conditioning count must raise an error.
</details>

<details><summary><b>Checkpoint 3</b></summary>
$(2/5)(3/7)=6/35$. The event accumulated before the next bar must have positive probability.
</details>

<details><summary><b>Checkpoint 4</b></summary>
$P(\text{alarm})=0.90(0.02)+0.05(0.98)$, faulty-and-alarm plus sound-and-alarm.
</details>

<details><summary><b>Checkpoint 5</b></summary>
$0.90(0.02)/[0.90(0.02)+0.05(0.98)]\approx0.269$, closer to 50% than to 2% or 90%; it is nevertheless far below the 90% sensitivity because sound parts dominate the population.
</details>

<details><summary><b>Checkpoint 6</b></summary>
The common full-sample denominator cancels between the two mask means. A zero mask raises `ValueError`; zero is a probability, while this conditional is undefined.
</details>

<details><summary><b>Checkpoint 7</b></summary>
Sensitivity is $P(+\mid D)$, not $P(D\mid +)$. We also need $P(D)$ and $P(+\mid D^c)$.
</details>

<details><summary><b>Checkpoint 8</b></summary>
The posterior falls because the true-positive route shrinks. The denominator is $0.92(0.004)+0.03(0.996)$.
</details>
